# 📓 공모펀드 EDA (2차) — 식별 코드 매칭과 NaN 의미

`public_funds` 테이블을 **처음부터 다시** 봅니다. 1차(`public_funds_eda.ipynb`)와의 차이:

- **DB가 `dtype=str` 로 재구축되어 선행 0 이 복구**되었습니다 — `or_co_xtn_itt_cd = '00040010'` 확인.
  1차 EDA 는 `40010.0` 을 보고 있었으므로 코드 구조 분석을 전부 다시 해야 합니다.
- 셀은 **관측 근거를 출력**할 뿐 판정하지 않습니다. 판정·검증 결과는 노트북 밖에 따로 기록합니다.

> **노트북 = 작업장 / 결론 기록 = 별도 문서** (1차와 동일한 규칙)

## 구성

| # | 주제 |
| :-: | :--- |
| 1 | **코드별 매칭** — 같은 종목을 여러 코드로 지칭하는가 · NaN 이 미부여인지 결측인지 |
| … | (이후 주제는 뒤에 추가) |


In [36]:
# ═══════════════════════════════════════════════════════════
# 안전 셋업 — 반드시 먼저 실행
# ═══════════════════════════════════════════════════════════
import sqlite3, re, sys
from pathlib import Path
import pandas as pd

DOMAIN = "public_funds"

ROOT = Path.cwd()
if not (ROOT / "data").exists():          # notebooks/ 안에서 열었을 때
    ROOT = ROOT.parent
DB = (ROOT / "data" / "financial_products.db").resolve()
assert DB.exists(), f"DB 없음: {DB}\n먼저 `python scripts/build_db.py` 실행"

# 🔒 읽기 전용 커넥션 — 쓰기 시도는 커넥션 레벨에서 거부됩니다
conn = sqlite3.connect(f"{DB.as_uri()}?mode=ro", uri=True)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}" if abs(v) < 1e6 else f"{v:,.0f}")

# 쓰기가 실제로 막혔는지 확인 (안 막혔으면 여기서 멈춤)
try:
    conn.execute("CREATE TABLE __probe__(x)")
    raise RuntimeError("❌ 쓰기가 허용됩니다 — 커넥션 설정을 확인하세요!")
except sqlite3.OperationalError as e:
    assert "readonly" in str(e), e
    print(f"🔒 읽기 전용 확인 — 원본 훼손 불가 ({e})")

TOTAL = conn.execute(f"SELECT COUNT(*) FROM {DOMAIN}").fetchone()[0]
COLS = [r[1] for r in conn.execute(f'PRAGMA table_info("{DOMAIN}")')]
print(f"📊 {DOMAIN}: {TOTAL:,}행 × {len(COLS)}컬럼")


def q(sql, params=()):
    """SQL 실행 → DataFrame"""
    return pd.read_sql_query(sql, conn, params=params)


🔒 읽기 전용 확인 — 원본 훼손 불가 (attempt to write a readonly database)
📊 public_funds: 95,619행 × 45컬럼


---

# 1. 코드별 매칭 — 같은 종목을 여러 코드로 지칭하는가

## 보는 컬럼 9종 (부여 주체별)

| 주체 | 컬럼 | 무엇 |
| :--- | :--- | :--- |
| 표준 | `itm_no` · `std_itm_no` | 종목(클래스)번호 · 표준종목번호 |
| 자산운용사 | `mtco_itm_no` | 운용사종목번호 — **모펀드 번호** |
| 한국예탁결제원 | `ksd_itm_no` · `rptt_ksd_itm_no` | 결제·보관용 번호 · 여러 클래스의 **대표** 번호 |
| 금융감독원 | `fss_itm_no` | 등록·감독용 번호 |
| 금융투자협회 | `kofia_fd_ccd` | 펀드**분류**코드 — 어떤 분류체계인지 확인 대상 |
| (기관 식별) | `or_co_xtn_itt_cd` · `trusc_xtn_itt_cd` | 운용회사 · 수탁회사 기관코드 |

## 답을 만들 두 질문

1. **매칭 관계** — 각 코드가 어느 단위(클래스 / 모펀드 / 기관)를 식별하고, `itm_no` 와
   1:1 인지 N:1 인지. 사용자가 어떤 코드로 물어도 같은 종목에 도달하는 경로가 있는가.
2. **NaN 의미** — 빈 값이 **미부여**(그 기관 체계에 원래 없음)인지 **결측**(있어야 하는데 빠짐)인지.
   특히 여러 코드가 **동시에** 비어 있을 때, 그 비어있음 자체가 어떤 그룹을 가리키는지.

## 셀 구성

| 셀 | 내용 |
| :- | :--- |
| ① 준비 · 종목 축약 | 적재 + '실질 빈 값' 작업 정의 + 종목 안에서 코드가 하나로 고정되는지 검증 후 축약 |
| ② 결측 분해 | NULL / 공백 / 더미 2종 / 오염 2종 — "비어 있음"의 진짜 크기 |
| ③ 코드 ↔ itm_no 대응 | 1:1 / N:1 + 값당 종목수 — 각 코드의 식별 단위 |
| ③-B 역방향 별칭 | 한 종목의 코드 집합 · 그 코드로 조회했을 때 종목이 특정되는가 |
| ③-C 코드값 공간 | 값·형태가 코드끼리 겹치는가 — 들어온 문자열을 어느 체계로 읽을지 |
| **④ 부여 주체별 매칭** | 도메인 §1.2 역할 분리를 축으로, 기관마다 무엇을 어느 단위로 부여했는지 |
| ④-A 운용사 | `or_co` × `mtco` — mtco 는 운용사 내부 번호. **(운용사, mtco) 합성키**가 모펀드의 진짜 키인가 |
| ④-B 예탁원 | `ksd` · `rptt` — 접두 5자리가 운용사와 맞물리는가 · 예탁원 묶음 vs 운용사 모펀드 묶음 |
| ④-C 금감원 | `fss` — 등록 단위가 클래스인가 펀드인가 · 앞 7자리의 정체 |
| ④-E 수탁사 | `trusc` — **펀드당 수탁사 1** 도메인 불변식 검증 · 운용사↔수탁사 N:M |
| ⑤ 코드 형태 | 길이·문자 마스크 · 기관코드 8 = 종별4 + 번호4 분해 |
| ⑥ 동시 결측 | 빈 값이 함께 움직이는가 → 그 그룹의 다른 컬럼 프로파일 |
| ⑥-B 코드별 결측 프로파일 | 코드마다 '값 있음' vs '값 없음' 을 같은 지표로 대조 |
| ⑥-C 실제 예시 | 값이 있을 때·없을 때의 진짜 종목과 원값 |
| ⑥-D 판매중 결측 | 결측이 판매상태로 설명되는 코드 vs 다른 이유가 있는 코드 |
| ⑦ 증거 요약표 | 관측치 한 표 |
| ⑧ 계층 요약 | 기관 → 펀드 → 클래스 를 앞 셀 실측값으로 한 장에 |

> ④ 는 **부여 주체(기관)** 를 축으로 봅니다. 식별자 문자열을 잘라 서로 유도하는
> 파싱 규칙을 만들려는 게 아니라, *"어느 기관이 무엇을 어느 단위로 부여했고,
> 그 단위가 운용사→모펀드→클래스 구조와 맞물리는가"* 를 확인하는 게 목적입니다.

> 🔻 **주제 1에서 제외한 축 (2026-08-17)** — 아래 둘은 *조인 경로*를 만들지 않아 이 주제 밖입니다.
> · **④-D 협회 축** — `kofia_fd_ccd` 는 식별자가 아니라 **분류 코드**입니다. 자리별 인코딩 해독은
>   "분류체계" 주제입니다. (확정분: 같은 모펀드 안에서 45.9% 가 갈림 → **클래스 수준 속성**.
>   6·10·11번째 자리가 갈리는 자리이나 **무엇을 인코딩하는지는 미해독**.)
> · **④-F 모자형 축** — 근거가 코드가 아니라 **종목명 파싱**이라 구조 분류 주제입니다.
>   (확정분: 모투자신탁·모투자회사 **0건** — 모펀드는 판매 상품이 아니므로 이 테이블에 없음.
>   `(or_co, mtco)` 묶음은 모자형이 아니라 **한 펀드의 클래스 묶음**.)
>
> 두 축의 결론·미해결 과제는 `docs/eda/public_funds_codes_handoff.md` §4.3 §4.6 §5 에 남아 있습니다.
> 라벨 A·B·C·E 는 그 문서와 대조할 수 있게 **재번호하지 않았습니다**.


In [37]:
# ── ① 준비 · 행 → 종목 축약 ────────────────────────────────────────
# 코드 9종 적재, '실질 빈 값' 작업 정의, 종목(itm_no) 단위 축약.
# 이후 셀들은 여기서 만든 item / ctx / eff_empty 를 사용합니다.

CODE_COLS = {
    "itm_no":           "종목번호 — 클래스(판매 단위)",
    "std_itm_no":       "표준종목번호",
    "ksd_itm_no":       "예탁원종목번호 — 결제·보관",
    "rptt_ksd_itm_no":  "대표예탁원번호 — 클래스들의 대표",
    "fss_itm_no":       "금감원종목번호 — 등록·감독",
    "mtco_itm_no":      "운용사종목번호 — 모펀드",
    "kofia_fd_ccd":     "금투협펀드분류코드",
    "or_co_xtn_itt_cd": "운용회사 기관코드",
    "trusc_xtn_itt_cd": "수탁회사 기관코드",
}
KNAME = {c: n.split(" — ")[0] for c, n in CODE_COLS.items()}   # 표에 같이 띄울 짧은 한글명
# 동시 결측 그룹의 성격을 대조할 맥락 컬럼
CTX_COLS = ["sale_yn", "or_attr_desc", "prvo_pbff_desc", "zrin_fd_ivst_risk_gcd", "fd_nast_suma"]

raw = q(f'SELECT {", ".join(list(CODE_COLS) + CTX_COLS)} FROM {DOMAIN}')
codes = raw[list(CODE_COLS)].apply(lambda s: s.astype("string").str.strip())


def eff_empty(s):
    """실질 빈 값(작업 정의) — NULL · 공백 · 0패딩 `(KR)?0+` · 한 문자 반복 `(.)\1+`.
    반복 패턴은 '9999999999' '111111111111' '11' 'PP' 류를 잡습니다 (스캔으로 실재 확인).
    탐지용 정의일 뿐, '미부여/결측' 판정이 아닙니다.
    ⚠️ 비영숫자 오염('"0466580')과 길이 이탈은 여기서 제외하지 않습니다 —
    따옴표 제거·0채움으로 복구 가능한 정보라서, ②에서 따로 셉니다."""
    x = s.fillna("")
    return (x == "") | x.str.fullmatch(r"(KR)?0+").fillna(False) | x.str.fullmatch(r"(.)\1+").fillna(False)


# 행 = 종목 × 속성태그(8.6배 부풀림). 축약 전에, 종목 안에서 코드 값이
# 정말 하나로 고정되는지부터 검증합니다 — 0 이 아니면 first() 가 정보를 숨깁니다.
g = codes.groupby("itm_no")
incons = {c: int((g[c].nunique(dropna=True) > 1).sum()) for c in CODE_COLS if c != "itm_no"}
print(f"종목 수 {g.ngroups:,} (행 {len(codes):,} → 종목당 평균 {len(codes)/g.ngroups:.1f}행)")
print(f"종목 내에서 값이 2개 이상인 코드 (전부 0 이어야 축약 가능): {incons}")

item = g.first()                                    # 종목 단위 (그룹의 첫 non-null)
item["itm_no"] = pd.Series(item.index, index=item.index, dtype="string")   # 인덱스로 빠진 키 복원
ctx = raw[CTX_COLS].groupby(codes["itm_no"]).first()
n_items = len(item)
print(f"→ 이후 셀은 종목 단위 {n_items:,}개 기준")


종목 수 11,139 (행 95,619 → 종목당 평균 8.6행)
종목 내에서 값이 2개 이상인 코드 (전부 0 이어야 축약 가능): {'std_itm_no': 0, 'ksd_itm_no': 0, 'rptt_ksd_itm_no': 0, 'fss_itm_no': 0, 'mtco_itm_no': 0, 'kofia_fd_ccd': 0, 'or_co_xtn_itt_cd': 0, 'trusc_xtn_itt_cd': 0}
→ 이후 셀은 종목 단위 11,139개 기준


In [38]:
# ── ② 실질 결측 분해 — NULL / 공백 / 더미 2종 / 오염 2종 ────────────
# "비어 있음"의 진짜 크기. NULL 만 세면 더미가 값처럼 보입니다.
#   더미  : 0패딩 (KR)?0+  ·  한 문자 반복 (.)\1+  ('9999999999' '11' 'PP' …)
#   오염  : 비영숫자 포함('"0466580') · 최빈 길이 이탈  → 셈만 하고 제외하지 않음
#           (따옴표 제거·0채움으로 복구 가능한 정보일 수 있어서. 판정은 별도 기록)

tbl = []
for c in CODE_COLS:
    s = item[c]
    filled = s[s.notna() & (s != "")]
    zero = filled.str.fullmatch(r"(KR)?0+").fillna(False)
    rept = filled.str.fullmatch(r"(.)\1+").fillna(False) & ~zero
    nonaln = ~filled.str.fullmatch(r"[0-9A-Za-z]+").fillna(False)
    clean = filled[~zero & ~rept]                    # 실질값만으로 길이 기준을 잡음
    modal = int(clean.str.len().mode().iloc[0]) if len(clean) else 0
    lenoff = int((clean.str.len() != modal).sum()) if len(clean) else 0
    em = eff_empty(s)
    tbl.append({
        "코드": c,
        "한글명": KNAME[c],
        "NULL": int(s.isna().sum()),
        "공백": int((s == "").sum()),
        "0패딩": int(zero.sum()),
        "반복문자": int(rept.sum()),
        "실질빈값 합": f"{int(em.sum()):,} ({em.mean()*100:.1f}%)",
        "오염:비영숫자": int(nonaln.sum()),
        "최빈길이": modal,
        "길이이탈": lenoff,
        "더미 값 예": dict(pd.concat([filled[zero], filled[rept]]).value_counts().head(3)),
    })

In [39]:
# print(f"종목 단위 {n_items:,}개 기준 · 실질빈값 = NULL+공백+0패딩+반복문자 (③⑦의 '값보유 종목'이 이걸 제외한 수)")
# print("※ 길이이탈 주의 — mtco_itm_no 는 선행 0 손실로 길이가 흔들리는 것이라 결측이 아님 (④ 가설 B 참조).")
# print("   trusc 7자리 14건 · ksd 'PP' '12' 류 · rptt 'KR5000000' 은 개별 확인 대상.")

pd.DataFrame(tbl)

,코드,한글명,NULL,공백,0패딩,반복문자,실질빈값 합,오염:비영숫자,최빈길이,길이이탈,더미 값 예
0,itm_no,종목번호,0,0,0,0,0 (0.0%),1,12,1,{}
1,std_itm_no,표준종목번호,9,0,0,0,9 (0.1%),0,12,0,{}
2,ksd_itm_no,예탁원종목번호,47,0,0,1,48 (0.4%),0,12,3,{'PP': 1}
3,rptt_ksd_itm_no,대표예탁원번호,9,0,478,0,487 (4.4%),0,12,2,"{'KR0000000000': 284, '000000000000': 192, 'KR000000000'..."
4,fss_itm_no,금감원종목번호,1,0,3022,2,"3,025 (27.2%)",0,12,0,"{'000000000000': 3022, '111111111111': 2}"
5,mtco_itm_no,운용사종목번호,1,0,33,28,62 (0.6%),8,7,2772,"{'00': 30, '9999999999': 14, '99999999': 6}"
6,kofia_fd_ccd,금투협펀드분류코드,12,0,2879,0,"2,891 (26.0%)",0,20,1,{'00000000000000000000': 2879}
7,or_co_xtn_itt_cd,운용회사 기관코드,9,0,0,0,9 (0.1%),0,8,0,{}
8,trusc_xtn_itt_cd,수탁회사 기관코드,9,0,0,0,9 (0.1%),0,8,14,{}


In [40]:
# ── ③ 각 코드 ↔ itm_no 대응 — 이 코드는 어느 단위를 식별하는가 ──────
# (종목 단위, 더미 제외)

tbl = []
for c in [c for c in CODE_COLS if c != "itm_no"]:
    s = item[c]
    s_clean = s[~eff_empty(s)]
    grp = s_clean.groupby(s_clean).size()
    n_shared = int((grp > 1).sum())
    # 더미를 '값'으로 조인하면 몇 종목이 한 값에 뭉치는지 (조인 사고의 크기)
    d = s[s.notna() & (s != "")]
    d = d[d.str.fullmatch(r"(KR)?0+").fillna(False)]
    tbl.append({
        "코드": c,
        "한글명": KNAME[c],
        "값보유 종목": len(s_clean),
        "distinct": int(grp.size),
        "값당 종목수": round(float(grp.mean()), 2) if len(grp) else None,
        "2종목+ 공유값": n_shared,
        "최대 공유": int(grp.max()) if len(grp) else 0,
        "관계(관측)": "1:1" if n_shared == 0 else ("거의 1:1" if n_shared < grp.size * 0.05 else "N:1 묶음"),
        "더미에 뭉친 종목": int(d.groupby(d).size().max()) if len(d) else 0,
    })

In [41]:
print("※ 1:1 = 클래스 단위 후보 / N:1 = 상위 묶음(모펀드·기관 등) 후보 — 관측 명명일 뿐 확정 아님")
pd.DataFrame(tbl)


※ 1:1 = 클래스 단위 후보 / N:1 = 상위 묶음(모펀드·기관 등) 후보 — 관측 명명일 뿐 확정 아님


,코드,한글명,값보유 종목,distinct,값당 종목수,2종목+ 공유값,최대 공유,관계(관측),더미에 뭉친 종목
0,std_itm_no,표준종목번호,11130,11127,1.000,3,2,거의 1:1,0
1,ksd_itm_no,예탁원종목번호,11091,11091,1.000,0,1,1:1,0
2,rptt_ksd_itm_no,대표예탁원번호,10652,2624,4.060,1877,15,N:1 묶음,284
3,fss_itm_no,금감원종목번호,8114,8084,1.000,25,4,거의 1:1,3022
4,mtco_itm_no,운용사종목번호,11077,4651,2.380,1643,19,N:1 묶음,30
5,kofia_fd_ccd,금투협펀드분류코드,8248,4781,1.730,1491,49,N:1 묶음,2879
6,or_co_xtn_itt_cd,운용회사 기관코드,11130,67,166.120,65,2852,N:1 묶음,0
7,trusc_xtn_itt_cd,수탁회사 기관코드,11130,18,618.330,16,2124,N:1 묶음,0


In [42]:
# ── ③-B 역방향 — 한 종목이 여러 코드로 불릴 때, 코드로 종목이 특정되는가 ──
# ③ 은 "코드값 하나가 몇 종목을 덮는가"(포함관계)를 봤습니다. 여기는 반대 방향입니다.
#   ① 한 종목이 몇 개의 이름(코드)을 갖는가
#   ② 사용자가 그 이름 중 하나를 던졌을 때 종목이 하나로 좁혀지는가
# 기관코드 2종(운용사·수탁사)은 종목이 아니라 기관을 가리키므로 제외합니다.

ITEM_CODES = [c for c in CODE_COLS if c not in ("or_co_xtn_itt_cd", "trusc_xtn_itt_cd")]

have = sum((~eff_empty(item[c])).astype(int) for c in ITEM_CODES)
print(f"① 별칭 집합 크기 — 종목 코드 {len(ITEM_CODES)}종 중 몇 개를 보유하는가")
for k, v in have.value_counts().sort_index().items():
    print(f"     {k}개 보유  {v:6,}종목")
print(f"   평균 {have.mean():.2f}개 · {len(ITEM_CODES)}개 전부 보유 "
      f"{int((have == len(ITEM_CODES)).sum()):,}종목 ({(have == len(ITEM_CODES)).mean()*100:.1f}%)")

print("\n② 역방향 유일성 — 이 코드로 조회했을 때 종목이 몇 개 나오는가")
rev = []
for c in ITEM_CODES:
    s = item[c][~eff_empty(item[c])]
    g = s.groupby(s).size()
    multi = g[g > 1]
    rev.append({
        "코드": c,
        "한글명": KNAME[c],
        "값 종류": int(g.size),
        "2종목+ 가리키는 값": int(multi.size),
        "그 값들이 덮는 종목": int(multi.sum()),
        "최대 종목수": int(g.max()) if len(g) else 0,
        "값보유 종목": int(s.size),
    })


① 별칭 집합 크기 — 종목 코드 7종 중 몇 개를 보유하는가
     1개 보유       1종목
     2개 보유      11종목
     3개 보유      45종목
     4개 보유     464종목
     5개 보유   2,358종목
     6개 보유     173종목
     7개 보유   8,087종목
   평균 6.41개 · 7개 전부 보유 8,087종목 (72.6%)

② 역방향 유일성 — 이 코드로 조회했을 때 종목이 몇 개 나오는가


In [43]:
print("※ '2종목+ 가리키는 값' 이 0 이면 그 코드 하나로 종목이 특정됩니다.")
print("   0 이 아니면 조회 결과가 여러 건이라는 관측 — 어떻게 응답할지는 노트북 밖에서 정합니다.")
pd.DataFrame(rev)


※ '2종목+ 가리키는 값' 이 0 이면 그 코드 하나로 종목이 특정됩니다.
   0 이 아니면 조회 결과가 여러 건이라는 관측 — 어떻게 응답할지는 노트북 밖에서 정합니다.


,코드,한글명,값 종류,2종목+ 가리키는 값,그 값들이 덮는 종목,최대 종목수,값보유 종목
0,itm_no,종목번호,11139,0,0,1,11139
1,std_itm_no,표준종목번호,11127,3,6,2,11130
2,ksd_itm_no,예탁원종목번호,11091,0,0,1,11091
3,rptt_ksd_itm_no,대표예탁원번호,2624,1877,9905,15,10652
4,fss_itm_no,금감원종목번호,8084,25,55,4,8114
5,mtco_itm_no,운용사종목번호,4651,1643,8069,19,11077
6,kofia_fd_ccd,금투협펀드분류코드,4781,1491,4958,49,8248


In [44]:
# ── ③-C 코드값 공간 — 들어온 문자열을 어느 체계로 읽을 것인가 ─────────
# ⑤ 는 코드별 형태를 봤고, 여기는 그 형태·값이 코드끼리 겹치는지를 봅니다.
# 겹치면 문자열만 보고 어느 컬럼의 값인지 가릴 수 없다는 관측입니다.
import itertools

def _val_map(col):
    """코드값 → 그 값을 가진 종목(itm_no) 집합. item 은 itm_no 가 인덱스이자
    컬럼이라 groupby(컬럼명) 은 모호합니다 — 값 Series 로 직접 묶습니다."""
    s = item[col][~eff_empty(item[col])]
    return s.groupby(s).apply(lambda g: set(g.index))

VAL = {c: _val_map(c) for c in ITEM_CODES}

print("① 값 공간 중첩 — 같은 문자열이 두 코드에 동시에 존재하는가")
found = False
for a, b in itertools.combinations(ITEM_CODES, 2):
    inter = set(VAL[a].index) & set(VAL[b].index)
    if not inter:
        continue
    found = True
    same = sum(1 for x in inter if VAL[a][x] == VAL[b][x])
    print(f"   {KNAME[a]}({a}) ∩ {KNAME[b]}({b}) = {len(inter)}건"
          f"  · 같은 종목 {same} / 다른 종목 {len(inter) - same}")
    for x in list(inter)[:2]:
        print(f"       {x!r}  {a}→{sorted(VAL[a][x])[:2]}  {b}→{sorted(VAL[b][x])[:2]}")
if not found:
    print("   겹치는 값 없음")

print("\n② 형태 중첩 — 마스크(숫자→9·영문→A)가 여러 코드에 공통인가")
mask = {}
for c in ITEM_CODES:
    for v in VAL[c].index:
        m = re.sub(r"[0-9]", "9", re.sub(r"[A-Za-z]", "A", v))
        mask.setdefault(m, set()).add(c)
shared = {m: cs for m, cs in mask.items() if len(cs) > 1}
print(f"   마스크 {len(mask)}종 · 여러 코드가 공유하는 마스크 {len(shared)}종")
for m, cs in sorted(shared.items(), key=lambda kv: (-len(kv[1]), kv[0]))[:8]:
    print(f"      {m:22s} → {sorted(cs)}")


① 값 공간 중첩 — 같은 문자열이 두 코드에 동시에 존재하는가
   표준종목번호(std_itm_no) ∩ 금감원종목번호(fss_itm_no) = 5건  · 같은 종목 5 / 다른 종목 0
       'K55301CC8722'  std_itm_no→['KR5153420267']  fss_itm_no→['KR5153420267']
       'K55226BQ9883'  std_itm_no→['KR5132450019']  fss_itm_no→['KR5132450019']
   예탁원종목번호(ksd_itm_no) ∩ 금감원종목번호(fss_itm_no) = 3건  · 같은 종목 3 / 다른 종목 0
       'KRZ501784621'  ksd_itm_no→['KR5119520006']  fss_itm_no→['KR5119520006']
       'KRZ502004762'  ksd_itm_no→['KR5153490136']  fss_itm_no→['KR5153490136']
   대표예탁원번호(rptt_ksd_itm_no) ∩ 금감원종목번호(fss_itm_no) = 1건  · 같은 종목 0 / 다른 종목 1
       '031910481390'  rptt_ksd_itm_no→['KR513509011M', 'KR513509012M']  fss_itm_no→['KR5153400000']

② 형태 중첩 — 마스크(숫자→9·영문→A)가 여러 코드에 공통인가
   마스크 130종 · 여러 코드가 공유하는 마스크 9종
      AA9999999999           → ['itm_no', 'ksd_itm_no', 'std_itm_no']
      AAA999999999           → ['fss_itm_no', 'ksd_itm_no', 'std_itm_no']
      9999                   → ['ksd_itm_no', 'mtco_itm_no']
      99999                  → ['ksd_itm_no', 'mt

In [45]:
# ── ④-A 운용사 축 — or_co_xtn_itt_cd × mtco_itm_no ─────────────────
# 도메인(§1.2): 운용을 지시하는 주체는 운용사이고, mtco 는 그 운용사 '내부' 관리번호.
# 내부 번호라면 전역 유일이 아닐 수 있다 → 모펀드의 진짜 키는 (운용사, mtco) 합성.

mz = item["mtco_itm_no"].str.zfill(7)                        # 선행 0 손실 보정
ok_fund = ~eff_empty(item["or_co_xtn_itt_cd"]) & ~eff_empty(item["mtco_itm_no"])
fund_key = (item["or_co_xtn_itt_cd"] + "|" + mz).where(ok_fund)   # 이후 셀에서 재사용

a = item[ok_fund]
scope = a.groupby(mz[ok_fund])["or_co_xtn_itt_cd"].nunique()
print(f"운용사·모펀드 모두 유효: {int(ok_fund.sum()):,}종목")
print(f"  mtco 값 {len(scope):,}종 중 여러 운용사에 걸침: {int((scope > 1).sum()):,}종 (최대 {int(scope.max())}개 운용사)")
print(f"  → mtco 단독 distinct {mz[ok_fund].nunique():,}  vs  (운용사,mtco) 합성 {fund_key.nunique():,}"
      f"   차이 {fund_key.nunique() - mz[ok_fund].nunique():,}개 모펀드가 단독 키에서 뭉쳐 있었음")
if (scope > 1).sum():
    ex = scope[scope > 1].index[:3]
    print(f"\n  걸침 예시 — 같은 mtco 를 서로 다른 운용사가 사용 (조합만 표시):")
    print(a.assign(mz=mz)[["mz", "or_co_xtn_itt_cd"]][mz[ok_fund].isin(ex)]
          .drop_duplicates().sort_values(["mz", "or_co_xtn_itt_cd"]).to_string(index=False))

sz = fund_key.dropna().groupby(fund_key.dropna()).size()
print(f"\n  모펀드당 클래스 수: 평균 {sz.mean():.2f} · 중앙 {int(sz.median())} · 최대 {int(sz.max())}"
      f" · 1클래스뿐 {int((sz == 1).sum()):,}개 ({(sz == 1).mean()*100:.1f}%)")
print(f"  운용사별 모펀드 수 상위 5: { {k: int(v) for k, v in a.groupby('or_co_xtn_itt_cd').apply(lambda gg: mz[gg.index].nunique()).sort_values(ascending=False).head(5).items()} }")

운용사·모펀드 모두 유효: 11,069종목
  mtco 값 4,573종 중 여러 운용사에 걸침: 65종 (최대 3개 운용사)
  → mtco 단독 distinct 4,573  vs  (운용사,mtco) 합성 4,643   차이 70개 모펀드가 단독 키에서 뭉쳐 있었음

  걸침 예시 — 같은 mtco 를 서로 다른 운용사가 사용 (조합만 표시):
     mz or_co_xtn_itt_cd
0002221         00040035
0002221         00080052
0003251         00040018
0003251         00080052
0003320         00040027
0003320         00080052

  모펀드당 클래스 수: 평균 2.38 · 중앙 1 · 최대 20 · 1클래스뿐 2,989개 (64.4%)
  운용사별 모펀드 수 상위 5: {'00080008': 1529, '00040007': 264, '00040010': 256, '00080035': 209, '00040011': 205}


In [46]:
# ── ④-B 예탁원 축 — ksd_itm_no · rptt_ksd_itm_no ───────────────────
# 도메인(§1.2): 예탁결제원은 증권 결제·보관을 위해 번호를 부여. rptt 는 "여러 클래스의 대표".
# 물음 ① rptt 앞 5자리는 예탁원이 매긴 운용사 번호인가 (운용사 축과 맞물리는가)
# 물음 ② 예탁원의 '대표' 묶음이 운용사의 모펀드 묶음과 같은 펀드를 가리키는가

b = item[~eff_empty(item["rptt_ksd_itm_no"]) & ~eff_empty(item["or_co_xtn_itt_cd"])]
pfx = b["rptt_ksd_itm_no"].str[:5]
p2o = pfx.groupby(pfx).apply(lambda s: b.loc[s.index, "or_co_xtn_itt_cd"].nunique())
o2p = b.groupby("or_co_xtn_itt_cd")["rptt_ksd_itm_no"].apply(lambda s: s.str[:5].nunique())
print("① rptt 앞 5자리 ↔ 운용사 대응")
print(f"   접두 {len(p2o)}종: 운용사 1개에만 대응 {int((p2o == 1).sum())}종 / 여러 운용사 {int((p2o > 1).sum())}종")
print(f"   운용사 {len(o2p)}종: 접두 1개 {int((o2p == 1).sum())}종 / 여러 접두 {int((o2p > 1).sum())}종 (합병·이관 이력이면 정상)")
if int((p2o > 1).sum()):
    print(f"   여러 운용사에 걸친 접두: { {k: int(v) for k, v in p2o[p2o > 1].items()} }")
if int((o2p > 1).sum()):
    print(f"   접두 여러 개인 운용사: { {k: int(v) for k, v in o2p[o2p > 1].items()} }")

print("\n② 예탁원 대표 묶음 vs 운용사 모펀드 묶음 — 같은 클래스들을 한 펀드로 묶는가")
d = pd.DataFrame({"r": item["rptt_ksd_itm_no"], "f": fund_key})
d = d[~eff_empty(d["r"]) & d["f"].notna()]
sz_r = d.groupby("r")["f"].transform("size")
sz_f = d.groupby("f")["r"].transform("size")
sz_j = d.groupby(["r", "f"])["r"].transform("size")
agree = (sz_r == sz_j) & (sz_f == sz_j)
print(f"   두 축 모두 유효: {len(d):,}종목 · 예탁원 묶음 {d['r'].nunique():,}개 / 운용사 묶음 {d['f'].nunique():,}개")
print(f"   두 묶음이 완전히 같은 종목: {int(agree.sum()):,} ({agree.mean()*100:.1f}%)")
print(f"   → 어긋나는 {int((~agree).sum()):,}종목은 두 기관의 '펀드' 경계가 다르다는 뜻 (어느 쪽이 틀린 게 아님)")
mix = d[~agree].groupby("r")["f"].nunique()
print(f"   예탁원 묶음 1개에 운용사 모펀드 2개 이상: {int((mix > 1).sum()):,}개 (최대 {int(mix.max()) if len(mix) else 0}개)")

print("\n③ ksd_itm_no 접두 — 예탁원 자체 대역")
k = item["ksd_itm_no"][~eff_empty(item["ksd_itm_no"])]
print(f"   앞 3자리 분포: { {kk: int(v) for kk, v in k.str[:3].value_counts().head(5).items()} }")

① rptt 앞 5자리 ↔ 운용사 대응
   접두 69종: 운용사 1개에만 대응 66종 / 여러 운용사 3종
   운용사 65종: 접두 1개 62종 / 여러 접두 3종 (합병·이관 이력이면 정상)
   여러 운용사에 걸친 접두: {'03228': 2, '03253': 2, '03258': 2}
   접두 여러 개인 운용사: {'00040011': 2, '00080008': 4, '00080052': 4}

② 예탁원 대표 묶음 vs 운용사 모펀드 묶음 — 같은 클래스들을 한 펀드로 묶는가
   두 축 모두 유효: 10,612종목 · 예탁원 묶음 2,610개 / 운용사 묶음 4,225개
   두 묶음이 완전히 같은 종목: 7,218 (68.0%)
   → 어긋나는 3,394종목은 두 기관의 '펀드' 경계가 다르다는 뜻 (어느 쪽이 틀린 게 아님)
   예탁원 묶음 1개에 운용사 모펀드 2개 이상: 560개 (최대 13개)

③ ksd_itm_no 접두 — 예탁원 자체 대역
   앞 3자리 분포: {'KRZ': 10992, 'KR7': 70, 'KR5': 26, '12': 1, '123': 1}


In [47]:
# ── ④-C 금감원 축 — fss_itm_no ─────────────────────────────────────
# 도메인(§1.2): 금감원은 펀드를 '등록·감독'. 등록 단위가 클래스인지 펀드인지가 관건.
# ⑤ 마스크에서 9999999A9999 형태(7자리 + 문자 + 4자리)가 지배적이었음 → 자리 분해로 확인.

f = item["fss_itm_no"][~eff_empty(item["fss_itm_no"])]
std = f[f.str.len() == 12]
print(f"금감원번호 보유 {len(f):,}종목 (12자리 {len(std):,})")
print(f"  8번째 자리 문자 분포: { {k: int(v) for k, v in std.str[7].value_counts().head(6).items()} }")
print(f"  앞 7자리 distinct {std.str[:7].nunique():,} · 뒤 4자리 distinct {std.str[8:].nunique():,}")

# 앞 7자리가 무엇의 단위인가 — 운용사/모펀드와 대조
p7 = std.str[:7]
cmp = pd.DataFrame({"p7": p7, "or_co": item.loc[std.index, "or_co_xtn_itt_cd"], "f": fund_key.loc[std.index]})
cmp = cmp[~eff_empty(cmp["or_co"])]
print(f"  앞 7자리 1개당 운용사 수: 1개뿐 {int((cmp.groupby('p7')['or_co'].nunique() == 1).sum()):,}"
      f" / 여럿 {int((cmp.groupby('p7')['or_co'].nunique() > 1).sum()):,}"
      f"   → 앞 7자리가 운용사 단위면 대부분 1개여야 함")
print(f"  앞 7자리 1개당 모펀드 수: 평균 {cmp.groupby('p7')['f'].nunique().mean():.2f}")

print("\n공유 fss — 같은 번호를 여러 클래스가 쓰는 경우 (등록 단위가 펀드로 내려간 흔적?)")
vc = f.groupby(f).size()
shared = vc[vc > 1]
inside = sum(1 for kk in shared.index if fund_key[f[f == kk].index].nunique(dropna=True) == 1)
print(f"  공유값 {len(shared)}종 · 총 {int(shared.sum())}종목")
print(f"    같은 모펀드 내부 공유: {inside}종 → 펀드 단위 등록으로 설명됨")
print(f"    서로 다른 모펀드 간 : {len(shared) - inside}종 → 개별 확인 대상 (오염 후보)")
if len(shared) - inside:
    for kk in shared.index:
        if fund_key[f[f == kk].index].nunique(dropna=True) > 1:
            print(f"      {kk}: {list(f[f == kk].index)}")

금감원번호 보유 8,114종목 (12자리 8,114)
  8번째 자리 문자 분포: {'C': 8027, 'M': 76, 'U': 2, '8': 2, '0': 2, '3': 1}
  앞 7자리 distinct 81 · 뒤 4자리 distinct 5,205
  앞 7자리 1개당 운용사 수: 1개뿐 75 / 여럿 6   → 앞 7자리가 운용사 단위면 대부분 1개여야 함
  앞 7자리 1개당 모펀드 수: 평균 33.58

공유 fss — 같은 번호를 여러 클래스가 쓰는 경우 (등록 단위가 펀드로 내려간 흔적?)
  공유값 25종 · 총 55종목
    같은 모펀드 내부 공유: 22종 → 펀드 단위 등록으로 설명됨
    서로 다른 모펀드 간 : 3종 → 개별 확인 대상 (오염 후보)
      0010175C4319: ['KR511704019M', 'KR5118430013']
      0010186C0412: ['KR5117651005', 'KR512102006M']
      0010205C4752: ['KR5153490530', 'KR5153490555']


In [48]:
# ── ④-E 수탁사 축 — trusc_xtn_itt_cd ───────────────────────────────
# 도메인(§1.2): 운용사가 망해도 자산은 수탁사에 남는다 — 자산 보관 주체는 펀드당 하나여야 함.
# 이건 법이 강제하는 구조라 '데이터 불변식'으로 검증 가능.

t = item["trusc_xtn_itt_cd"]
ok = ~eff_empty(t) & fund_key.notna()
per_fund = t[ok].groupby(fund_key[ok]).nunique()
bad = per_fund[per_fund > 1]
print(f"불변식 검증 — 모펀드 하나에 수탁사 하나인가")
print(f"  모펀드 {len(per_fund):,}개 중 수탁사 2개 이상: {len(bad):,}개 ({len(bad)/len(per_fund)*100:.2f}%)")
print(f"  → 도메인 불변식이 데이터에서 성립. 위반은 이관(수탁사 변경) 이력 또는 오염 후보")
if len(bad):
    for fkey in bad.index[:5]:
        idx = fund_key[fund_key == fkey].index
        print(f"    {fkey}: { {k: int(v) for k, v in t[idx].value_counts().items()} }")

print("\n운용사 ↔ 수탁사 관계 (N:M — 한 운용사가 여러 수탁사를 쓰는가)")
o = item["or_co_xtn_itt_cd"]
ok2 = ~eff_empty(t) & ~eff_empty(o)
per_or = t[ok2].groupby(o[ok2]).nunique()
per_tr = o[ok2].groupby(t[ok2]).nunique()
print(f"  운용사 {len(per_or)}개: 수탁사 평균 {per_or.mean():.1f}개 (최대 {int(per_or.max())})")
print(f"  수탁사 {len(per_tr)}개: 운용사 평균 {per_tr.mean():.1f}개 (최대 {int(per_tr.max())})")
print(f"  종별(앞4) 분포: { {k: int(v) for k, v in t[ok2].str[:4].value_counts().items()} }  ← 0002 대부분 = 은행")

불변식 검증 — 모펀드 하나에 수탁사 하나인가
  모펀드 4,643개 중 수탁사 2개 이상: 9개 (0.19%)
  → 도메인 불변식이 데이터에서 성립. 위반은 이관(수탁사 변경) 이력 또는 오염 후보
    00040001|0214201: {'00020088': 5, '00020081': 1}
    00040010|02ETF34: {'00020054': 1, '00020027': 1}
    00040010|02M7500: {'00020020': 3, '00020032': 2}
    00040018|00AG530: {'00020032': 11, '00020004': 1}
    00040027|0001843: {'00020081': 5, '00020004': 1}

운용사 ↔ 수탁사 관계 (N:M — 한 운용사가 여러 수탁사를 쓰는가)
  운용사 67개: 수탁사 평균 5.2개 (최대 13)
  수탁사 18개: 운용사 평균 19.4개 (최대 46)
  종별(앞4) 분포: {'0002': 10753, '0005': 336, '0016': 41}  ← 0002 대부분 = 은행


In [49]:
# ── ⑤ 코드 형태 — 길이·문자 구조 마스크 ────────────────────────────
# 숫자→9, 대문자→A 로 치환한 마스크의 상위 3종. 자리 구조가 눈에 보입니다.
# (종목 단위, 더미 제외)

for c in CODE_COLS:
    s = item[c]
    s = s[~eff_empty(s)]
    m = s.str.replace(r"[0-9]", "9", regex=True).str.replace(r"[A-Z]", "A", regex=True)
    top = m.value_counts().head(3)
    print(f"{c:18s} {KNAME[c]:<10s} " + "   ".join(f"{k} ×{v:,}" for k, v in top.items()))

print("\n기관코드 8자리 = 종별(4) + 기관번호(4) 분해:")
for c in ["or_co_xtn_itt_cd", "trusc_xtn_itt_cd"]:
    s = item[c]
    s = s[~eff_empty(s)]
    print(f"  {c:18s} ({KNAME[c]}) 종별 { {k: int(v) for k, v in s.str[:4].value_counts().items()} }")
    print(f"  {' ':18s} 번호부(뒤4) distinct {s.str[-4:].nunique()} · 전체(8자리) distinct {s.nunique()}"
          f" → 종별이 다른데 번호부가 같은 조합이 {s.nunique() - s.str[-4:].nunique()}건")


itm_no             종목번호       AA9999999999 ×8,927   AA999999999A ×2,137   AA999A999999 ×72
std_itm_no         표준종목번호     A99999AA9999 ×3,858   AA9999999999 ×3,165   A99999A99999 ×1,929
ksd_itm_no         예탁원종목번호    AAA999999999 ×8,799   AAA99999999A ×2,192   AA9999999999 ×79
rptt_ksd_itm_no    대표예탁원번호    999999999999 ×7,249   999999999A99 ×1,091   9999999AA999 ×686
fss_itm_no         금감원종목번호    9999999A9999 ×7,961   9999999A99AA ×75   9999999A999A ×34
mtco_itm_no        운용사종목번호    9999999 ×5,416   999999 ×1,337   9999A99 ×972
kofia_fd_ccd       금투협펀드분류코드  99999999999999999999 ×1,767   99999999999999999AA9 ×1,167   9999999999999999999A ×1,060
or_co_xtn_itt_cd   운용회사 기관코드  99999999 ×11,130
trusc_xtn_itt_cd   수탁회사 기관코드  99999999 ×11,116   9999999 ×14

기관코드 8자리 = 종별(4) + 기관번호(4) 분해:
  or_co_xtn_itt_cd   (운용회사 기관코드) 종별 {'0004': 5773, '0008': 5357}
                     번호부(뒤4) distinct 55 · 전체(8자리) distinct 67 → 종별이 다른데 번호부가 같은 조합이 12건
  trusc_xtn_itt_cd   (수탁회사 기관코드) 종별 {'0002': 10753, '000

In [50]:
# ── ⑥ 동시 결측 — 빈 값이 함께 움직이는가, 그 그룹은 어떤 펀드인가 ──
# 여러 코드가 같이 비면 "적재 누락"으로 단정하지 않고, 그 패턴 그룹이
# 다른 컬럼(판매상태·평가·순자산)에서도 구분되는지 대조합니다 —
# 비어있음 자체가 정보일 수 있습니다.

LET = dict(zip(CODE_COLS, "ISKRFMCOT"))
E = pd.DataFrame({c: eff_empty(item[c]) for c in CODE_COLS}, index=item.index)
patt = E.apply(lambda r: "".join(l if r[c] else "·" for c, l in LET.items()), axis=1)
pc = patt.value_counts()
print("문자 = 그 코드가 '실질 빈 값' / · = 값 있음")
print("순서: " + "  ".join(f"{l}={c}" for c, l in LET.items()))
print()
print(pc.head(10).to_string())

print(f"\n상위 패턴별 프로파일 — 빈 값 그룹이 다른 컬럼에서도 구분되는가:")
prof = []
for p in pc.head(6).index:
    cc = ctx[patt == p]
    mo = cc["or_attr_desc"].astype("string").str.strip().mode()
    prof.append({
        "패턴": p,
        "종목": int((patt == p).sum()),
        "판매중%": round(float((cc["sale_yn"].astype("string").str.strip() == "판매중").mean()) * 100, 1),
        "위험등급보유%": round(float(cc["zrin_fd_ivst_risk_gcd"].notna().mean()) * 100, 1),
        "순자산보유%": round(float(cc["fd_nast_suma"].notna().mean()) * 100, 1),
        "최다유형": mo.iloc[0] if len(mo) else "—",
    })


문자 = 그 코드가 '실질 빈 값' / · = 값 있음
순서: I=itm_no  S=std_itm_no  K=ksd_itm_no  R=rptt_ksd_itm_no  F=fss_itm_no  M=mtco_itm_no  C=kofia_fd_ccd  O=or_co_xtn_itt_cd  T=trusc_xtn_itt_cd

·········    8087
····F·C··    2357
···RF·C··     431
····F····     154
····FMC··      32
··KRF·C··      26
······C··      13
··KRFMC··      11
···RFMC··      10
·SKR··COT       8

상위 패턴별 프로파일 — 빈 값 그룹이 다른 컬럼에서도 구분되는가:


In [51]:
print("※ 읽는 법 — 특정 코드'만' 비는 패턴 = 그 기관 체계에서의 미부여 후보 /")
print("   여러 코드가 같이 비는 패턴 = 비어있음 자체가 한 그룹(예: 평가 중단·판매 종료)일 가능성.")
print("   프로파일이 다른 패턴과 뚜렷이 다르면 '의미 있는 비어있음'.")
pd.DataFrame(prof)

※ 읽는 법 — 특정 코드'만' 비는 패턴 = 그 기관 체계에서의 미부여 후보 /
   여러 코드가 같이 비는 패턴 = 비어있음 자체가 한 그룹(예: 평가 중단·판매 종료)일 가능성.
   프로파일이 다른 패턴과 뚜렷이 다르면 '의미 있는 비어있음'.


,패턴,종목,판매중%,위험등급보유%,순자산보유%,최다유형
0,·········,8087,90.200,90.300,98.500,주식형
1,····F·C··,2357,42.000,45.000,48.000,주식형
2,···RF·C··,431,0.000,6.300,1.600,채권형
3,····F····,154,93.500,94.200,98.100,채권혼합
4,····FMC··,32,15.600,15.600,15.600,주식형
5,··KRF·C··,26,0.000,7.700,0.000,MMF


In [52]:
# ── ⑥-B 코드별 결측 프로파일 — 이 코드가 빌 때 그 종목은 어떤 그룹인가 ──
# ⑥ 은 결측 '패턴' 단위로 봤습니다. 여기는 '코드' 단위로, 값이 있는 종목과
# 없는 종목을 같은 지표로 나란히 놓습니다. 두 줄이 비슷하면 비어있음이 무작위고,
# 다르면 비어있음이 특정 그룹을 가리킨다는 관측입니다.
# ⚠️ 여기서 나오는 건 대조표일 뿐입니다. '미부여'인지 '결측'인지의 판정은
#    도메인 근거와 함께 노트북 밖에서 정합니다 (작업규칙 1·3).

_sale = ctx["sale_yn"].astype("string").str.strip()
_attr = ctx["or_attr_desc"].astype("string").str.strip()

miss_prof = []
for c in [c for c in CODE_COLS if c != "itm_no"]:
    em = eff_empty(item[c])
    for label, sel in [("값 있음", ~em), ("값 없음", em)]:
        if not int(sel.sum()):
            continue
        mo = _attr[sel].mode()
        miss_prof.append({
            "코드": c,
            "한글명": KNAME[c],
            "구분": label,
            "종목": int(sel.sum()),
            "판매중%": round(float((_sale[sel] == "판매중").mean()) * 100, 1),
            "위험등급보유%": round(float(ctx.loc[sel, "zrin_fd_ivst_risk_gcd"].notna().mean()) * 100, 1),
            "순자산보유%": round(float(ctx.loc[sel, "fd_nast_suma"].notna().mean()) * 100, 1),
            "최다유형": mo.iloc[0] if len(mo) else "—",
        })


In [53]:
print("※ 같은 코드의 '값 있음' / '값 없음' 두 줄을 비교하세요.")
print("   지표가 비슷하면 비어있음이 무작위, 뚜렷이 다르면 비어있음이 한 그룹을 가리킵니다.")
pd.DataFrame(miss_prof)


※ 같은 코드의 '값 있음' / '값 없음' 두 줄을 비교하세요.
   지표가 비슷하면 비어있음이 무작위, 뚜렷이 다르면 비어있음이 한 그룹을 가리킵니다.


,코드,한글명,구분,종목,판매중%,위험등급보유%,순자산보유%,최다유형
0,std_itm_no,표준종목번호,값 있음,11130,75.900,77.000,83.400,주식형
1,std_itm_no,표준종목번호,값 없음,9,0.000,11.100,100.000,—
2,ksd_itm_no,예탁원종목번호,값 있음,11091,76.100,77.200,83.700,주식형
3,ksd_itm_no,예탁원종목번호,값 없음,48,0.000,14.600,18.800,MMF
4,rptt_ksd_itm_no,대표예탁원번호,값 있음,10652,79.300,80.100,87.000,주식형
5,rptt_ksd_itm_no,대표예탁원번호,값 없음,487,0.000,7.200,3.900,채권형
6,fss_itm_no,금감원종목번호,값 있음,8114,90.000,90.200,98.500,주식형
7,fss_itm_no,금감원종목번호,값 없음,3025,37.700,41.200,43.000,주식형
8,mtco_itm_no,운용사종목번호,값 있음,11077,76.100,77.200,83.800,주식형
9,mtco_itm_no,운용사종목번호,값 없음,62,16.400,25.800,22.600,주식형


In [54]:
# ── ⑥-C 실제 예시 — 값이 있을 때와 없을 때의 진짜 종목 ────────────────
# 표의 숫자만 보면 무엇이 비었는지 감이 안 옵니다. 실제 행을 꺼내 봅니다.

_nm = q(f"SELECT itm_no, itm_nm FROM {DOMAIN}")
_nm = _nm.apply(lambda s: s.astype("string").str.strip()).groupby("itm_no").first()
name = _nm["itm_nm"].reindex(item.index).fillna("")

for c in [c for c in CODE_COLS if c != "itm_no"]:
    em = eff_empty(item[c])
    print(f"\n▸ {c} ({KNAME[c]}) — 실질 빈값 {int(em.sum()):,}종목")
    ok = item.loc[~em, c]
    if len(ok):
        print("   값 있음 예:")
        for i in ok.index[:2]:
            print(f"      {i}  {ok[i]!r:<24s} {name[i][:44]}")
    if int(em.sum()):
        print("   값 없음 예 (원값을 그대로 표시 — 더미인지 NULL인지 보이도록):")
        for i in item.index[em][:3]:
            raw_v = item.loc[i, c]
            shown = "NULL" if pd.isna(raw_v) else repr(raw_v)
            print(f"      {i}  {shown:<24s} {name[i][:44]}"
                  f"   [{_sale[i]}]")



▸ std_itm_no (표준종목번호) — 실질 빈값 9종목
   값 있음 예:
      KR5010101401  'KR5205104064'           서울신종MMF1
      KR5010101402  'KR5205104072'           서울신종MMF2
   값 없음 예 (원값을 그대로 표시 — 더미인지 NULL인지 보이도록):
      "  NULL                     공모   [<NA>]
      KR5153470111  NULL                     미래에셋글로벌포커스4.0마켓헤지증권자투자신탁(H)(주식-파생형) 종류C1   [판매완료]
      KR5153470112  NULL                     미래에셋글로벌포커스4.0마켓헤지증권자투자신탁(H)(주식-파생형) 종류C2   [판매완료]

▸ ksd_itm_no (예탁원종목번호) — 실질 빈값 48종목
   값 있음 예:
      KR5010101401  'KRZ500023820'           서울신종MMF1
      KR5010101402  'KRZ500023830'           서울신종MMF2
   값 없음 예 (원값을 그대로 표시 — 더미인지 NULL인지 보이도록):
      "  NULL                     공모   [<NA>]
      KR510901005M  NULL                     미래에셋디스커버리증권자투자신탁G1호(채권)(C-I)   [판매완료]
      KR511013548M  NULL                     미래에셋시스템캡안정혼합 1Y L-2호   [판매완료]

▸ rptt_ksd_itm_no (대표예탁원번호) — 실질 빈값 487종목
   값 있음 예:
      KR5010101404  '030410031114'           서울신종MMF  4
      KR5010101702  '031910230221'           미래에셋국공채MM

In [55]:
# ── ⑥-D 판매중인데도 비는 코드 — 결측이 판매상태로 설명되는가 ──────────
# ⑥-B 에서 '값 없음' 행의 판매중% 가 코드마다 크게 달랐습니다. 여기서 갈라 봅니다.
#   · 판매중 결측이 0 인 코드 → 결측이 판매상태만으로 설명됨
#   · 판매중에도 비는 코드   → 다른 이유가 있음 → 후보 축들과 대조
# ⚠️ 대조표일 뿐 원인 판정이 아닙니다. 판정은 노트북 밖(ontology/enums)에 기록합니다.

_on = _sale == "판매중"
_off = _sale == "판매완료"
print(f"판매중 {int(_on.sum()):,} · 판매완료 {int(_off.sum()):,} · 기타 {int((~_on & ~_off).sum()):,}")

split = []
for c in [c for c in CODE_COLS if c != "itm_no"]:
    em = eff_empty(item[c])
    split.append({
        "코드": c, "한글명": KNAME[c],
        "실질빈값": int(em.sum()),
        "판매중에서 빔": int((em & _on).sum()),
        "판매완료에서 빔": int((em & _off).sum()),
        "판매중 내 결측률%": round(float((em & _on).sum()) / int(_on.sum()) * 100, 2),
    })
print()
print(pd.DataFrame(split).to_string(index=False))

# 판매중에도 비는 코드만 후보 축과 대조
hot = [r["코드"] for r in split if r["판매중에서 빔"] > 0]
print(f"\n판매중에도 비는 코드: {hot}")

_ja = name.str.contains("자투자신탁|자투자회사|자투신", regex=True)
_etf = name.str.contains("상장지수", regex=False)
_tailA = item["itm_no"].str[-1].str.isalpha().fillna(False)

print("\n후보 축별 대조 — 판매중 종목만, '결측 있음' 비율")
for c in hot:
    em = eff_empty(item[c]) & _on
    print(f"\n  ▸ {c} ({KNAME[c]})  판매중 결측 {int(em.sum()):,}")
    for label, ax in [("자펀드", _ja), ("상장지수", _etf), ("종목번호 끝=영문", _tailA)]:
        a = ax & _on
        if not int(a.sum()):
            continue
        print(f"      {label:14s} 해당 {int(a.sum()):5,}종목 결측률 {em[a].mean()*100:5.1f}%"
              f"   |  비해당 {int((~ax & _on).sum()):5,}종목 결측률 {em[~ax & _on].mean()*100:5.1f}%")
    byor = pd.DataFrame({"or_co": item["or_co_xtn_itt_cd"], "em": em})[_on]
    g = byor.groupby("or_co")["em"].agg(["size", "mean"])
    g = g[g["size"] >= 50].sort_values("mean", ascending=False)
    if len(g):
        top = ", ".join(f"{k}:{v*100:.0f}%" for k, v in g["mean"].head(3).items())
        print(f"      운용사별(판매중 50종목+) 최고 {top}  ·  0% 인 운용사 {int((g['mean']==0).sum())}/{len(g)}곳")


판매중 8,445 · 판매완료 2,693 · 기타 0

              코드       한글명  실질빈값  판매중에서 빔  판매완료에서 빔  판매중 내 결측률%
      std_itm_no    표준종목번호     9        0         8       0.000
      ksd_itm_no   예탁원종목번호    48        0        47       0.000
 rptt_ksd_itm_no   대표예탁원번호   487        0       486       0.000
      fss_itm_no   금감원종목번호  3025     1139      1885      13.490
     mtco_itm_no   운용사종목번호    62       10        51       0.120
    kofia_fd_ccd 금투협펀드분류코드  2891     1004      1886      11.890
or_co_xtn_itt_cd 운용회사 기관코드     9        0         8       0.000
trusc_xtn_itt_cd 수탁회사 기관코드     9        0         8       0.000

판매중에도 비는 코드: ['fss_itm_no', 'mtco_itm_no', 'kofia_fd_ccd']

후보 축별 대조 — 판매중 종목만, '결측 있음' 비율

  ▸ fss_itm_no (금감원종목번호)  판매중 결측 1,139
      자펀드            해당 6,142종목 결측률  17.9%   |  비해당 2,303종목 결측률   1.7%
      종목번호 끝=영문      해당 1,157종목 결측률  36.7%   |  비해당 7,288종목 결측률   9.8%
      운용사별(판매중 50종목+) 최고 00080008:47%, 00040067:21%, 00080019:15%  ·  0% 인 운용사 4/29곳

  ▸ mtco_itm_no (운용사종목번호)  판매중 결측

In [56]:
# ── ⑦ 증거 요약표 — 관측치 한 표 ──────────────────────────────────

summ = []
for c, kname in CODE_COLS.items():
    s = item[c]
    em = eff_empty(s)
    s_clean = s[~em]
    grp = s_clean.groupby(s_clean).size()
    summ.append({
        "한글명": kname,
        "실질빈값": f"{int(em.sum()):,} ({em.mean()*100:.1f}%)",
        "distinct": int(grp.size),
        "값당 종목수": round(float(grp.mean()), 2) if len(grp) else None,
        "최대 공유": int(grp.max()) if len(grp) else 0,
    })
summary_df = pd.DataFrame(summ, index=list(CODE_COLS))
summary_df


,한글명,실질빈값,distinct,값당 종목수,최대 공유
itm_no,종목번호 — 클래스(판매 단위),0 (0.0%),11139,1.000,1
std_itm_no,표준종목번호,9 (0.1%),11127,1.000,2
ksd_itm_no,예탁원종목번호 — 결제·보관,48 (0.4%),11091,1.000,1
rptt_ksd_itm_no,대표예탁원번호 — 클래스들의 대표,487 (4.4%),2624,4.060,15
fss_itm_no,금감원종목번호 — 등록·감독,"3,025 (27.2%)",8084,1.000,4
mtco_itm_no,운용사종목번호 — 모펀드,62 (0.6%),4651,2.380,19
kofia_fd_ccd,금투협펀드분류코드,"2,891 (26.0%)",4781,1.730,49
or_co_xtn_itt_cd,운용회사 기관코드,9 (0.1%),67,166.120,2852
trusc_xtn_itt_cd,수탁회사 기관코드,9 (0.1%),18,618.330,2124


In [57]:
# ── ⑧ 계층 요약 — 기관 → 펀드 → 클래스 (전부 앞 셀의 실측값) ──────────
# ①~⑦ 에서 나온 수치만으로 계층을 한 장에 모읍니다. 새로 계산하지 않습니다.

_n_or = int(item.loc[~eff_empty(item["or_co_xtn_itt_cd"]), "or_co_xtn_itt_cd"].nunique())
_n_tr = int(item.loc[~eff_empty(item["trusc_xtn_itt_cd"]), "trusc_xtn_itt_cd"].nunique())
_n_fund = int(fund_key.nunique())
_n_rptt = int(item.loc[~eff_empty(item["rptt_ksd_itm_no"]), "rptt_ksd_itm_no"].nunique())
_n_item = len(item)
_sz = fund_key.dropna().groupby(fund_key.dropna()).size()

print(f"""
[기관]  운용사 {_n_or}개 (or_co_xtn_itt_cd)      수탁사 {_n_tr}개 (trusc_xtn_itt_cd)
             8자리 = 종별(4) + 번호(4)              8자리 = 종별(4) + 번호(4)
   │  1:N
   ▼
[펀드]  (or_co, mtco) 합성키 {_n_fund:,}개        ← mtco 단독은 {int(mz[ok_fund].nunique()):,}로 뭉침 (④-A 와 동일 기준)
        예탁원 대표 묶음 {_n_rptt:,}개 (rptt)      ← 경계가 다름 (④-B 참조)
   │  1:N  (펀드당 클래스 평균 {_sz.mean():.2f} · 1개뿐 {int((_sz==1).sum()):,}개)
   ▼
[클래스] itm_no {_n_item:,}개  ★ 속성이 붙는 주 노드
        1:1 동반 코드 — ksd_itm_no · std_itm_no · fss_itm_no
        N:1 묶음 코드 — rptt · mtco · kofia   (③-B 역방향 유일성 참조)
""")
print("※ 이 그림의 모든 숫자는 위 셀들의 출력에서 온 것입니다. 관계의 '의미'(왜 그런가)는")
print("   도메인 문서와 대조해 노트북 밖에서 기록합니다.")



[기관]  운용사 67개 (or_co_xtn_itt_cd)      수탁사 18개 (trusc_xtn_itt_cd)
             8자리 = 종별(4) + 번호(4)              8자리 = 종별(4) + 번호(4)
   │  1:N
   ▼
[펀드]  (or_co, mtco) 합성키 4,643개        ← mtco 단독은 4,573로 뭉침 (④-A 와 동일 기준)
        예탁원 대표 묶음 2,624개 (rptt)      ← 경계가 다름 (④-B 참조)
   │  1:N  (펀드당 클래스 평균 2.38 · 1개뿐 2,989개)
   ▼
[클래스] itm_no 11,139개  ★ 속성이 붙는 주 노드
        1:1 동반 코드 — ksd_itm_no · std_itm_no · fss_itm_no
        N:1 묶음 코드 — rptt · mtco · kofia   (③-B 역방향 유일성 참조)

※ 이 그림의 모든 숫자는 위 셀들의 출력에서 온 것입니다. 관계의 '의미'(왜 그런가)는
   도메인 문서와 대조해 노트북 밖에서 기록합니다.


---

# 2. 종목 자체 피처 — 고유값의 의미와 결측의 의미

코드 9종을 뺀 종목 속성들을 봅니다. 주제 1과 문제가 다릅니다 —
**결측률은 대부분 0.1% 미만이고, 문제는 "이 값이 무슨 뜻인가"** 입니다.

## 대상 (미판정 18종을 세 블록으로)

| 블록 | 대상 | 성격 |
| :--- | :--- | :--- |
| **A** | 저distinct 분류 15종 (distinct 1~11) | 값 전수 나열이 가능 → 도메인 문서·정답 100건과 대조 |
| **B** | `prfd_attr_cd` (228종 · 15축 EAV) | 한 컬럼에 여러 종류가 섞임 → 형태로 갈라야 함 |
| **C** | `bmrk_nm` · `bmrk_eng_nm` (391종) | 자유 텍스트 → **이번엔 고유값만** 정리 (ETF 대조는 나중) |

## 축약 제약 ★

```
블록 A · C  →  종목(itm_no) 단위 축약 가능
블록 B      →  🔴 축약 불가. prfd_attr_cd 는 PK 구성요소라 종목 안에서 갈린다
                (종목당 4~16개 태그 · 행 95,619 기준으로 다뤄야 함)
```

45컬럼 중 종목 안에서 값이 갈리는 것은 `prfd_attr_cd` **하나뿐**입니다.

## 셀 구성

| 셀 | 내용 |
| :- | :--- |
| ⓪ 준비 | 주제 2 컬럼 적재 · 종목 내 값 고정 검증 후 축약 · `prfd_attr_cd` 는 행 단위 별도 |
| A-1 | 값 전수 — 도메인 축(§4)별로 묶어 모든 고유값과 빈도 |
| A-2 | 이상징후 점검 — 상수 컬럼 · 코드↔명 불일치 · `해당없음` 비율 · 정답 100건 검증 가능 여부 |
| B-1 | `prfd_attr_cd` 형태로 갈래 나누기 — 국가형 / 코드형 / 기타 |
| B-2 | **A와의 겹침 검사** — 값 집합이 겹치는가 · 같은 것을 두 번 말하는가 |
| C-1 | 벤치마크 고유값 정리 (판정·파싱 없음) |

> 주제 1과 같은 규칙: 셀은 **관측 근거만 출력**하고 판정하지 않습니다.
> 판정은 `ontology/enums/public_funds.yaml` 과 `docs/eda/public_funds_codes_conclusions.md` 에 기록합니다.


In [64]:
# ── 주제2 ⓪ 준비 — 종목 자체 피처 적재 ─────────────────────────────
# 도메인 문서 §4 의 묶음을 그대로 축으로 씁니다 (작업규칙 4).
# ①의 item 은 코드 9종 + 맥락 5컬럼만 담고 있어 재사용이 안 됩니다.

FEAT_AXES = {
    "운용·투자 속성": {
        "or_attr_desc":       "운용속성구분코드 설명",
        "fd_set_pcd":         "펀드설정유형코드",
    },
    "지역·환율": {
        "ovrs_fd_desc":       "해외펀드구분코드 설명",
        "fd_ivst_rgn_desc":   "펀드투자지역구분코드 설명",
        "fd_estb_ctry_cd":    "펀드설립국가코드",
        "curr_cd":            "통화코드",
    },
    "가입·판매": {
        "prvo_pbff_desc":     "사모/공모구분코드 설명",
        "pfiv_sale_cntl_tcd": "전문투자자판매제어구분코드",
        "sale_yn":            "판매여부",
        "thco_sale_yn":       "당사판매여부",
        "ofsfd_yn":           "역외펀드여부",
    },
    "위험": {
        "zrin_fd_ivst_risk_gcd":    "제로인위험등급코드",
        "zrin_fd_ivst_risk_grd_nm": "제로인위험등급명",
    },
    "과세·기타": {
        "int_dvd_desc":       "이자배당구분코드 설명",
        "frc_bpr_itm_yn":     "외화기준가종목여부",
        "hdge_fd_yn":         "헤지펀드여부",
    },
}
FEAT_A = {c: k for ax in FEAT_AXES.values() for c, k in ax.items()}
FEAT_C = {"bmrk_nm": "벤치마크명", "bmrk_eng_nm": "벤치마크영문명"}
KNAME2 = {**FEAT_A, **FEAT_C}

_cols = list(FEAT_A) + list(FEAT_C)
raw2 = q(f'SELECT itm_no, {", ".join(_cols)} FROM {DOMAIN}')
raw2 = raw2.apply(lambda s: s.astype("string").str.strip())

# ①과 같은 안전장치 — 종목 안에서 값이 정말 하나로 고정되는지 먼저 확인
g2 = raw2.groupby("itm_no")
inc2 = {c: int((g2[c].nunique(dropna=True) > 1).sum()) for c in _cols}
bad2 = {c: v for c, v in inc2.items() if v}
print(f"종목 {g2.ngroups:,} · 축약 대상 {len(_cols)}컬럼")
print(f"  종목 내에서 값이 2개 이상인 컬럼 (전부 0 이어야 축약 가능): {bad2 if bad2 else '없음 ✅'}")
item2 = g2.first()

# 🔴 prfd_attr_cd 는 종목 안에서 갈리므로 축약하지 않고 행 단위로 둡니다
attr = q(f"SELECT itm_no, prfd_attr_cd FROM {DOMAIN}")
attr = attr.apply(lambda s: s.astype("string").str.strip())
_per = attr.groupby("itm_no")["prfd_attr_cd"].nunique()
print(f"  prfd_attr_cd — 행 {len(attr):,} · 값 {attr['prfd_attr_cd'].nunique()}종 "
      f"· 종목당 태그 {_per.min()}~{_per.max()}개 (평균 {_per.mean():.2f})  ← 축약하지 않음")


종목 11,139 · 축약 대상 18컬럼
  종목 내에서 값이 2개 이상인 컬럼 (전부 0 이어야 축약 가능): 없음 ✅
  prfd_attr_cd — 행 95,619 · 값 228종 · 종목당 태그 1~16개 (평균 8.58)  ← 축약하지 않음


In [59]:
# ── 주제2 A-1 값 전수 — 저distinct 분류 컬럼의 모든 고유값 ──────────
# distinct 가 최대 11이라 전수 나열이 가능합니다. 표본이 아니라 전량입니다.
# 도메인 문서 §4 의 축으로 묶어 출력합니다.

for axis, cols in FEAT_AXES.items():
    print(f"\n{'═'*70}\n▣ {axis}\n{'═'*70}")
    for c, kname in cols.items():
        s = item2[c]
        vc = s.value_counts(dropna=False)
        nn = int(s.isna().sum())
        print(f"\n  {c}  ({kname})   distinct {s.nunique()} · NULL {nn:,}({nn/len(s)*100:.1f}%)")
        for v, n in vc.items():
            isna = pd.isna(v)                      # pd.NA 는 v != v 로 못 잡습니다
            shown = "NULL" if isna else repr(v)
            tag = "  ← 결측" if isna else ""
            print(f"       {shown:<28s} {n:6,}종목 ({n/len(s)*100:5.1f}%){tag}")



══════════════════════════════════════════════════════════════════════
▣ 운용·투자 속성
══════════════════════════════════════════════════════════════════════

  or_attr_desc  (운용속성구분코드 설명)   distinct 11 · NULL 9(0.1%)
       '주식형'                         4,280종목 ( 38.4%)
       '재간접'                         3,022종목 ( 27.1%)
       '채권형'                         1,182종목 ( 10.6%)
       '채권혼합'                        1,154종목 ( 10.4%)
       '06'                            686종목 (  6.2%)
       '주식혼합'                          358종목 (  3.2%)
       'MMF'                           250종목 (  2.2%)
       '혼합자산'                          133종목 (  1.2%)
       '특별자산'                           48종목 (  0.4%)
       '임대형'                            15종목 (  0.1%)
       NULL                              9종목 (  0.1%)  ← 결측
       '대출형'                             2종목 (  0.0%)

  fd_set_pcd  (펀드설정유형코드)   distinct 3 · NULL 1(0.0%)
       '10'                         10,469종목 ( 94.0%)
       '20'             

In [60]:
# ── 주제2 A-2 이상징후 점검 — 값을 그냥 믿기 전에 볼 것 ─────────────
# ⚠️ 판정이 아니라 '확인이 필요한 지점' 목록입니다.

chk = []
for c, kname in FEAT_A.items():
    s = item2[c]
    vc = s.value_counts()
    nn = int(s.isna().sum())
    top = vc.index[0] if len(vc) else None
    chk.append({
        "컬럼": c, "한글명": kname,
        "distinct": int(s.nunique()),
        "최빈값": repr(top) if top is not None else "—",
        "최빈비율%": round(float(vc.iloc[0]) / len(s) * 100, 1) if len(vc) else None,
        "NULL%": round(nn / len(s) * 100, 2),
        "상수?": "🔴 예" if s.nunique() <= 1 else "",
        "해당없음 포함?": "예" if (s == "해당없음").any() else "",
    })
print("① 컬럼별 요약 — '상수?' 가 예면 정보량이 0입니다")
_chk = pd.DataFrame(chk)

print("\n② 위험등급 코드 ↔ 명칭 대응 — 1:1 이어야 합니다")
z = item2[["zrin_fd_ivst_risk_gcd", "zrin_fd_ivst_risk_grd_nm"]].dropna()
pair = z.groupby("zrin_fd_ivst_risk_gcd")["zrin_fd_ivst_risk_grd_nm"].unique()
print(f"   코드 {item2['zrin_fd_ivst_risk_gcd'].nunique()}종 · 명칭 {item2['zrin_fd_ivst_risk_grd_nm'].nunique()}종")
for k, v in pair.items():
    mark = "  🔴 1코드에 여러 명칭" if len(v) > 1 else ""
    print(f"     {k!r:8s} → {list(v)}{mark}")
orphan = z.groupby("zrin_fd_ivst_risk_grd_nm")["zrin_fd_ivst_risk_gcd"].nunique()
if (orphan > 1).any():
    print(f"   🔴 1명칭에 여러 코드: { {k: int(v) for k, v in orphan[orphan>1].items()} }")

print("\n③ 정답 100건으로 검증 가능한 컬럼 (schema.xlsx Sheet2_Sample 의 axis_* 와 대응)")
print("   or_attr_desc      → axis_fundType            (대조 완료: 12/12 순도)")
print("   prvo_pbff_desc    → axis_investorEligibility (대조 완료: 3/3 순도)")
print("   fd_set_pcd        → axis_issuanceType        (대조 미완: '10'→추가형 93/93, '20' 혼재)")
print("   ※ 대조 결과는 docs/eda/public_funds_codes_conclusions.md §7")
_chk


① 컬럼별 요약 — '상수?' 가 예면 정보량이 0입니다

② 위험등급 코드 ↔ 명칭 대응 — 1:1 이어야 합니다
   코드 7종 · 명칭 9종
     '1.0'    → ['매우 높은 위험']
     '2.0'    → ['높은 위험', '높은위험']  🔴 1코드에 여러 명칭
     '20054.0' → ['06']
     '3.0'    → ['다소 높은 위험']
     '4.0'    → ['보통 위험', '보통위험']  🔴 1코드에 여러 명칭
     '5.0'    → ['낮은 위험']
     '6.0'    → ['매우 낮은 위험']

③ 정답 100건으로 검증 가능한 컬럼 (schema.xlsx Sheet2_Sample 의 axis_* 와 대응)
   or_attr_desc      → axis_fundType            (대조 완료: 12/12 순도)
   prvo_pbff_desc    → axis_investorEligibility (대조 완료: 3/3 순도)
   fd_set_pcd        → axis_issuanceType        (대조 미완: '10'→추가형 93/93, '20' 혼재)
   ※ 대조 결과는 docs/eda/public_funds_codes_conclusions.md §7


,컬럼,한글명,distinct,최빈값,최빈비율%,NULL%,상수?,해당없음 포함?
0,or_attr_desc,운용속성구분코드 설명,11,'주식형',38.400,0.080,,
1,fd_set_pcd,펀드설정유형코드,3,'10',94.000,0.010,,
2,ovrs_fd_desc,해외펀드구분코드 설명,3,'해외',48.100,0.080,,
3,fd_ivst_rgn_desc,펀드투자지역구분코드 설명,7,'국내',42.100,0.080,,
4,fd_estb_ctry_cd,펀드설립국가코드,2,'000',97.400,0.080,,
5,curr_cd,통화코드,2,'KRW',99.400,0.010,,
6,prvo_pbff_desc,사모/공모구분코드 설명,2,'공모',99.800,0.080,,
7,pfiv_sale_cntl_tcd,전문투자자판매제어구분코드,3,'00',99.300,0.080,,
8,sale_yn,판매여부,2,'판매중',75.800,0.010,,
9,thco_sale_yn,당사판매여부,2,'Y',93.800,6.230,,


In [61]:
# ── 주제2 B-1 prfd_attr_cd 형태로 갈래 나누기 ──────────────────────
# 도메인 §4.4: 한 컬럼에 '펀드별 속성 태그' 와 '투자 국가 태그' 가 섞여 있습니다.
# 의미를 모르는 상태에서 먼저 할 일은 형태로 갈래를 가르는 것입니다.
# 🔴 행 단위입니다 (종목 안에서 갈리므로 축약 불가).

av = attr["prfd_attr_cd"]
mask_a = av.str.replace(r"[0-9]", "9", regex=True).str.replace(r"[A-Za-z]", "A", regex=True)
print(f"행 {len(attr):,} · 값 {av.nunique()}종")
print("\n① 형태 마스크 분포")
for m, n in mask_a.value_counts().items():
    ex = list(av[mask_a == m].drop_duplicates()[:4])
    print(f"   {m:<8s} 행 {n:7,} · 값 {av[mask_a==m].nunique():4d}종   예: {ex}")

print("\n② 첫 글자별 — 축 후보")
first = av.str[0]
for ch, n in first.value_counts().items():
    sub = av[first == ch]
    print(f"   '{ch}'  행 {n:7,} · 값 {sub.nunique():4d}종   예: {list(sub.drop_duplicates()[:6])}")

print("\n③ 종목당 태그 조합 — 같은 축에서 하나만 고르는가, 여러 개 붙는가")
per_axis = attr.assign(ax=first).groupby(["itm_no", "ax"])["prfd_attr_cd"].nunique()
multi = per_axis[per_axis > 1]
print(f"   (종목, 첫글자) 조합 {len(per_axis):,} 중 값이 2개 이상: {len(multi):,}")
if len(multi):
    print(f"   그런 첫글자: { {k: int(v) for k, v in multi.reset_index().groupby('ax').size().items()} }")
    print("   → 값이 여러 개인 축은 '단일 선택' 이 아니라 다중 태그입니다")


행 95,619 · 값 228종

① 형태 마스크 분포
   A999     행  93,948 · 값  210종   예: ['F103', 'V102', 'M111', 'C103']
   AAA      행   1,670 · 값   17종   예: ['CHN', 'VNM', 'JPN', 'USA']
   해외       행       1 · 값    1종   예: ['해외']

② 첫 글자별 — 축 후보
   'C'  행  22,922 · 값    5종   예: ['C103', 'C101', 'C102', 'C104', 'CHN']
   'M'  행  21,072 · 값   17종   예: ['M111', 'M109', 'M112', 'M101', 'M113', 'M105']
   'D'  행  15,315 · 값    6종   예: ['D102', 'D106', 'D105', 'D101', 'D103', 'DEU']
   'V'  행  11,264 · 값    4종   예: ['V102', 'V101', 'V103', 'VNM']
   'G'  행   5,505 · 값   16종   예: ['G110', 'G118', 'G115', 'G119', 'G101', 'G102']
   'W'  행   5,359 · 값   35종   예: ['W101', 'W106', 'W104', 'W102', 'W142', 'W122']
   'N'  행   4,057 · 값   60종   예: ['N121', 'N177', 'N178', 'N122', 'N135', 'N142']
   'F'  행   3,958 · 값    8종   예: ['F103', 'F101', 'F102', 'F104', 'F106', 'FRA']
   'P'  행   1,708 · 값   10종   예: ['P101', 'P103', 'P104', 'P102', 'P121', 'P122']
   'S'  행   1,532 · 값   16종   예: ['S104', 'S108', 'S120', 'S107

In [62]:
# ── 주제2 B-2 A와의 겹침 검사 — 같은 것을 두 번 말하고 있는가 ────────
# ① 값 집합이 문자 그대로 겹치는가  ② 종목 단위로 같은 정보를 담는가

print("① 값 집합 겹침 — prfd_attr_cd 의 값이 A 컬럼의 값과 같은 문자열인가")
aset = set(av.dropna())
found = False
for c, kname in FEAT_A.items():
    vs = set(item2[c].dropna())
    inter = aset & vs
    if inter:
        found = True
        print(f"   {c} ({kname}) ∩ prfd_attr_cd = {len(inter)}개  {sorted(inter)[:8]}")
if not found:
    print("   겹치는 값 없음 — 두 축은 서로 다른 코드 공간을 씁니다")

print("\n② 정보 중복 — 국가형 태그 vs 지역/국가/통화 컬럼")
ctry = av[av.str.fullmatch(r"[A-Z]{3}").fillna(False)]
ctry_items = attr.loc[ctry.index, "itm_no"]
print(f"   국가형(AAA) 태그 보유 행 {len(ctry):,} · 종목 {ctry_items.nunique():,} "
      f"({ctry_items.nunique()/len(item2)*100:.1f}%) · 값 {ctry.nunique()}종")
print(f"   값: {sorted(ctry.unique())}")

tag1 = attr.loc[ctry.index].assign(tag=ctry.values).groupby("itm_no")["tag"].apply(
    lambda s: s.iloc[0] if s.nunique() == 1 else "복수")
for c in ["fd_ivst_rgn_desc", "fd_estb_ctry_cd", "curr_cd", "ovrs_fd_desc"]:
    joined = pd.DataFrame({"tag": tag1, "other": item2[c].reindex(tag1.index)}).dropna()
    if not len(joined):
        continue
    pure = sum(1 for _, gg in joined.groupby("tag") if gg["other"].nunique() == 1)
    print(f"\n   ▸ 국가태그 × {c} ({KNAME2.get(c, c)})   태그 {joined['tag'].nunique()}종 중 "
          f"상대 컬럼이 1값에만 대응 {pure}종")
    print(pd.crosstab(joined["tag"], joined["other"]).to_string())

print("\n   ※ 교차표가 대각선에 몰리면 같은 것의 세분, 흩어지면 서로 다른 축입니다.")
print("     도메인 §4.4 는 '지역 7종보다 세분된 국가 축' 이라고 적어두었습니다 — 여기서 확인됩니다.")


① 값 집합 겹침 — prfd_attr_cd 의 값이 A 컬럼의 값과 같은 문자열인가
   ovrs_fd_desc (해외펀드구분코드 설명) ∩ prfd_attr_cd = 1개  ['해외']

② 정보 중복 — 국가형 태그 vs 지역/국가/통화 컬럼
   국가형(AAA) 태그 보유 행 1,670 · 종목 1,549 (13.9%) · 값 17종
   값: ['AUS', 'BRA', 'CHN', 'DEU', 'ESP', 'FRA', 'GBR', 'HKG', 'IDN', 'IND', 'JPN', 'KOR', 'MYS', 'RUS', 'TWN', 'USA', 'VNM']

   ▸ 국가태그 × fd_ivst_rgn_desc (펀드투자지역구분코드 설명)   태그 16종 중 상대 컬럼이 1값에만 대응 7종
other  국내  글로벌  남미/북미  아시아  유럽  이머징/브릭스
tag                                    
AUS     0    0      0    6   0        0
BRA     0   20     26    0   0       10
CHN     8  186      0  343   0       22
DEU     0    3      0    0  15        0
ESP     0    3      0    0   0        0
FRA     0    3      0    0   0        0
GBR     0    3      0    0   0        0
IDN     0    0      0   13   0        0
IND     2   27      0   35   0       43
JPN     1   44      0   81   0        0
MYS     0    6      0    0   0        0
RUS     0    3      0   17  15        8
TWN     0    7      0    0   0        0
USA    

In [63]:
# ── 주제2 C-1 벤치마크 고유값 정리 (판정·파싱 없음) ─────────────────
# 목적: 나중에 ETF cu_base_index 와 대조할 목록을 만드는 것.
# 이번 셀에서는 값을 정리만 하고 의미 판정이나 파싱 규칙을 만들지 않습니다.

for c, kname in FEAT_C.items():
    s = item2[c]
    print(f"\n▸ {c} ({kname})   distinct {s.nunique():,} · NULL {int(s.isna().sum()):,}")

bm = item2["bmrk_nm"]
vc = bm.value_counts()
print(f"\n① 상위 25종 (전체 {len(vc):,}종)")
for v, n in vc.head(25).items():
    print(f"     {n:5,}종목  {v[:66]}")

print("\n② 형태 플래그 — 나중에 파싱할 때 쓸 분류 (지금은 세기만)")
flags = pd.DataFrame({
    "값": vc.index,
    "종목수": vc.values,
    "복합식(%포함)": [bool(pd.Series([v]).str.contains(r"\d+\s*%", regex=True).iloc[0]) for v in vc.index],
    "가산식(+포함)": ["+" in v for v in vc.index],
    "영문만": [bool(pd.Series([v]).str.fullmatch(r"[\x00-\x7F]+").iloc[0]) for v in vc.index],
})
print(f"   복합식(% 포함) {int(flags['복합식(%포함)'].sum()):,}종 · "
      f"가산식(+ 포함) {int(flags['가산식(+포함)'].sum()):,}종 · "
      f"영문만 {int(flags['영문만'].sum()):,}종")
print("\n   복합식 예:")
for v in flags[flags["복합식(%포함)"]]["값"].head(4):
    print(f"     {v[:80]}")

print("\n③ 전량 목록 — 아래 표. 파일로도 이미 있습니다:")
print("     ontology/enums/public_funds.bmrk_nm.values.txt")
print("     (ETF 대조 시 overseas_etfs.cu_base_index.values.txt · 국내ETF 와 17종 통용 — entity_map §2)")
flags



▸ bmrk_nm (벤치마크명)   distinct 391 · NULL 0

▸ bmrk_eng_nm (벤치마크영문명)   distinct 388 · NULL 0

① 상위 25종 (전체 391종)
     1,509종목  KOSPI200
       798종목  MSCI ACWI CR 50% + 종합채권01Y 50%
       550종목  KOSPI200 25% + 종합채권01Y 75%
       399종목  MSCI ACWI
       337종목  종합채권02Y 90%
       308종목  MSCI ACWI CR 25% + 종합채권01Y 75%
       285종목  채권종합지수 1~2년
       243종목  중소형지수
       217종목  CALL
       203종목  MSCI CHINA
       184종목  KAP CD 6개월  90%
       164종목  코스피 고배당 50
       163종목  제로인 대안투자기대수익지수
       157종목  KOSPI200 10% + 종합채권 01Y 90%
       145종목  KOSPI200 50% + 종합채권01Y 50%
       126종목  MSCI ACWI Information Technology
       123종목  종합채권 1~2년
       107종목  Bloomberg GLOBAL AGGREGATE(KRW HEDGED) 90%
       107종목  종합채권 3개월~1년
       103종목  MSCI AC ASIA PACIFIC ex JAPAN
       102종목  국공채02Y 90%
        99종목  KOSPI200 25% + KIS채권종합01Y 75%
        97종목  S&P 500
        94종목  MSCI INDIA
        94종목  MSCI ACWI Health Care

② 형태 플래그 — 나중에 파싱할 때 쓸 분류 (지금은 세기만)
   복합식(% 포함) 253종 · 가산식(+ 포함) 211종 · 영문만

,값,종목수,복합식(%포함),가산식(+포함),영문만
0,KOSPI200,1509,False,False,True
1,MSCI ACWI CR 50% + 종합채권01Y 50%,798,True,True,False
2,KOSPI200 25% + 종합채권01Y 75%,550,True,True,False
3,MSCI ACWI,399,False,False,True
4,종합채권02Y 90%,337,True,False,False
...,...,...,...,...,...
386,MSCI BRIC 40% + KIS국공채3Y 60%,1,True,True,False
387,FTSE Xinhua China25 Index 40% + 국공채3Y 60%,1,True,True,False
388,MSCI EM CR 40% + KIS채권종합 60%,1,True,True,False
389,RICI Agriculture TR 35% + 국공채03Y 60% + CD 5%,1,True,True,False
